# 🏎️ Análisis de Ventas — Concesionaria Multimarca 2022–2024
### Con Modelos ML para Predicción de Margen y Días de Cierre

**Autora:** Micaela Feriale — Analista de Datos  
**Stack:** Python · pandas · scikit-learn · matplotlib · seaborn

---
| | |
|---|---|
| 📦 Dataset | 4.000 ventas · 28 modelos · 8 vendedores · 2022–2024 |
| 💰 Revenue total | $65.47B en 3 años |
| 📊 EDA | 12 visualizaciones profesionales |
| 🤖 ML Target 1 | Predicción de **Margen %** por venta |
| 🤖 ML Target 2 | Predicción de **Días de Cierre** |


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

pd.set_option("display.float_format", "{:,.0f}".format)
PALETTE = ["#2563EB","#7C3AED","#059669","#D97706","#DC2626","#0891B2"]
print("✅ Setup completo")


## 2. Carga y Exploración

In [ ]:
df = pd.read_csv("../data/processed/ventas_clean.csv", parse_dates=["fecha"])
print(f"Dimensiones: {df.shape}")
print(f"Período: {df['fecha'].min().date()} → {df['fecha'].max().date()}")
print(f"\nKPIs principales:")
print(f"  Revenue total    : ${df['precio_total'].sum()/1e9:.2f}B")
print(f"  Ganancia neta    : ${df['ganancia_neta'].sum()/1e9:.2f}B")
print(f"  Margen promedio  : {df['margen_pct'].mean():.1f}%")
print(f"  Ticket promedio  : ${df['precio_final'].mean():,.0f}")
print(f"  Rating promedio  : {df['rating_cliente'].mean():.2f}/5")
df[["modelo","segmento","precio_final","margen_pct","ganancia_neta","vendedor","dias_cierre"]].head(8)


In [ ]:
# Distribución por segmento
seg = df.groupby("segmento").agg(
    ventas=("venta_id","count"),
    revenue=("precio_total","sum"),
    margen=("margen_pct","mean"),
    ticket=("precio_final","mean"),
    dias=("dias_cierre","mean")
).reset_index()
seg["revenue_share"] = seg["revenue"]/seg["revenue"].sum()*100
seg.style.background_gradient(cmap="Blues", subset=["revenue","margen"]).format({
    "revenue":"${:,.0f}","margen":"{:.1f}%","ticket":"${:,.0f}","revenue_share":"{:.1f}%","dias":"{:.1f}"
})


## 3. Análisis Temporal

In [ ]:
m = df.groupby("year_month").agg(
    ventas=("venta_id","count"),
    revenue=("precio_total","sum"),
    ganancia=("ganancia_neta","sum"),
    margen=("margen_pct","mean")
).reset_index()
m["mom"] = m["revenue"].pct_change()*100

fig, ax = plt.subplots(2,1,figsize=(14,8),gridspec_kw={"height_ratios":[3,1]})
fig.suptitle("Revenue y Ganancia Mensual 2022–2024",fontsize=14,fontweight="bold")
x = range(len(m))
ax[0].bar(x, m["revenue"]/1e9, color=PALETTE[0], alpha=0.75, label="Revenue (B$)")
ax2 = ax[0].twinx()
ax2.plot(x, m["ganancia"]/1e9, color=PALETTE[2], lw=2.5, marker="o", ms=5, label="Ganancia (B$)")
ax[0].set_ylabel("Revenue (B$)"); ax2.set_ylabel("Ganancia (B$)", color=PALETTE[2])
ax[0].set_xticks(x); ax[0].set_xticklabels(m["year_month"], rotation=45, ha="right", fontsize=7)
ax[0].grid(axis="y",alpha=0.4)
colors_m=[PALETTE[2] if v>=0 else PALETTE[4] for v in m["mom"].fillna(0)]
ax[1].bar(x, m["mom"].fillna(0), color=colors_m, alpha=0.8)
ax[1].axhline(0,color="black",lw=0.8); ax[1].set_ylabel("Var MoM (%)")
ax[1].set_xticks(x); ax[1].set_xticklabels(m["year_month"],rotation=45,ha="right",fontsize=7)
plt.tight_layout(); plt.show()


## 4. Análisis por Modelo y Vendedor

In [ ]:
# Top 10 modelos
top10 = df.groupby(["modelo","segmento"]).agg(
    ventas=("venta_id","count"), revenue=("precio_total","sum"),
    ganancia=("ganancia_neta","sum"), margen=("margen_pct","mean")
).reset_index().sort_values("revenue",ascending=False).head(10)

fig,axes = plt.subplots(1,2,figsize=(15,6))
fig.suptitle("Top 10 Modelos",fontsize=14,fontweight="bold")
seg_c = {"Económico":PALETTE[3],"Intermedio":PALETTE[0],"Premium":PALETTE[1],"Luxury":PALETTE[4]}
colors = [seg_c.get(s) for s in top10["segmento"]]
axes[0].barh(top10["modelo"][::-1], top10["revenue"][::-1]/1e9, color=colors[::-1], edgecolor="white")
axes[0].set_xlabel("Revenue (B$)"); axes[0].set_title("Revenue por Modelo")
axes[1].barh(top10["modelo"][::-1], top10["margen"][::-1], color=colors[::-1], alpha=0.8, edgecolor="white")
axes[1].set_xlabel("Margen (%)"); axes[1].set_title("Margen por Modelo")
plt.tight_layout(); plt.show()


In [ ]:
# Performance de vendedores
sel = df.groupby(["vendedor","zona","senioridad"]).agg(
    ventas=("venta_id","count"),
    revenue=("precio_total","sum"),
    ganancia=("ganancia_neta","sum"),
    margen=("margen_pct","mean"),
    dias=("dias_cierre","mean"),
    rating=("rating_cliente","mean"),
    comisiones=("comision_vendedor","sum")
).reset_index().sort_values("ganancia",ascending=False)

fig,axes = plt.subplots(2,2,figsize=(14,10))
fig.suptitle("Performance de Vendedores",fontsize=14,fontweight="bold")
axes[0,0].bar(sel["vendedor"],sel["ventas"],color=PALETTE[0],edgecolor="white",alpha=0.85)
axes[0,0].set_title("Ventas"); axes[0,0].tick_params(axis="x",rotation=30)
axes[0,1].bar(sel["vendedor"],sel["ganancia"]/1e6,color=PALETTE[2],edgecolor="white",alpha=0.85)
axes[0,1].set_title("Ganancia (M$)"); axes[0,1].tick_params(axis="x",rotation=30)
axes[1,0].bar(sel["vendedor"],sel["margen"],color=PALETTE[1],edgecolor="white",alpha=0.85)
axes[1,0].axhline(sel["margen"].mean(),color=PALETTE[4],linestyle="--",lw=1.3)
axes[1,0].set_title("Margen %"); axes[1,0].tick_params(axis="x",rotation=30)
axes[1,1].bar(sel["vendedor"],sel["dias"],color=PALETTE[3],edgecolor="white",alpha=0.85)
axes[1,1].set_title("Días Cierre Prom."); axes[1,1].tick_params(axis="x",rotation=30)
plt.tight_layout(); plt.show()
print(sel[["vendedor","ventas","ganancia","margen","dias","rating"]].to_string(index=False))


## 5. Heatmap Vendedor × Segmento

In [ ]:
pivot = df.pivot_table(index="vendedor",columns="segmento",
    values="ganancia_neta",aggfunc="sum")/1e6
fig,ax = plt.subplots(figsize=(11,6))
sns.heatmap(pivot,ax=ax,cmap="Blues",annot=True,fmt=".0f",
    linewidths=0.5,linecolor="#E2E8F0",cbar_kws={"label":"Ganancia (M$)"})
ax.set_title("Ganancia por Vendedor y Segmento (M$)",fontsize=14,fontweight="bold")
plt.tight_layout(); plt.show()


## 6. Financiación, Origen y Descuentos

In [ ]:
fin = df.groupby("financiacion").agg(
    ventas=("venta_id","count"),ticket=("precio_final","mean"),dias=("dias_cierre","mean")).reset_index()
orig = df.groupby("origen_lead").agg(
    ventas=("venta_id","count"),ticket=("precio_final","mean"),dias=("dias_cierre","mean")).reset_index()
disc = df.groupby("descuento_pct").agg(
    ventas=("venta_id","count"),margen=("margen_pct","mean"),dias=("dias_cierre","mean")).reset_index()

fig,axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle("Financiación, Origen y Descuentos",fontsize=14,fontweight="bold")
axes[0].bar(fin["financiacion"],fin["ventas"],color=PALETTE[:len(fin)],edgecolor="white")
axes[0].set_title("Ventas por Financiación"); axes[0].tick_params(axis="x",rotation=20)
axes[1].bar(orig["origen_lead"],orig["ventas"],color=PALETTE[:len(orig)],edgecolor="white",alpha=0.85)
axes[1].set_title("Ventas por Origen de Lead"); axes[1].tick_params(axis="x",rotation=20)
axes[2].plot(disc["descuento_pct"],disc["margen"],color=PALETTE[4],lw=2.5,marker="o",ms=7)
axes[2].set_xlabel("Descuento (%)"); axes[2].set_ylabel("Margen (%)"); axes[2].set_title("Descuento vs. Margen"); axes[2].grid(True,alpha=0.4)
plt.tight_layout(); plt.show()


## 7. Machine Learning — Predicción de Margen y Días de Cierre

In [ ]:
# Preparar features
df_ml = df.copy()
df_ml["permuta"] = df_ml["permuta"].astype(int)
df_ml["test_drive"] = df_ml["test_drive"].astype(int)

CAT = ["marca","tipo","segmento","financiacion","origen_lead","zona","senioridad"]
encoders = {}
for col in CAT:
    le = LabelEncoder()
    df_ml[col+"_enc"] = le.fit_transform(df_ml[col].astype(str))
    encoders[col] = le

FEATS = ["precio_lista","descuento_pct","num_accesorios","permuta","test_drive",
         "segmento_num","es_premium_luxury","mes","año"] + [c+"_enc" for c in CAT]
X = df_ml[[f for f in FEATS if f in df_ml.columns]]

def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/(y_true+1e-9)))*100
    return {"model":name,"MAE":round(mae,3),"R2":round(r2,4),"MAPE":round(mape,1)}

results_all = {}
for target in ["margen_pct","dias_cierre"]:
    y = df_ml[target]
    Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=42)
    res = []
    for name, mdl in [
        ("Ridge", Pipeline([("sc",StandardScaler()),("m",Ridge(alpha=5))])),
        ("Random Forest", RandomForestRegressor(200,max_depth=12,n_jobs=-1,random_state=42)),
        ("Gradient Boosting", GradientBoostingRegressor(250,max_depth=4,learning_rate=0.06,random_state=42)),
    ]:
        mdl.fit(Xtr,ytr)
        res.append(evaluate(yte,mdl.predict(Xte),name))
    results_all[target] = pd.DataFrame(res).sort_values("R2",ascending=False)
    print(f"\n── {target} ──")
    print(results_all[target].to_string(index=False))


In [ ]:
# Comparación visual
fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle("Comparación de Modelos ML",fontsize=14,fontweight="bold")
for ax,(target,rdf) in zip(axes,results_all.items()):
    colors_c = [PALETTE[2] if i==0 else PALETTE[0] for i in range(len(rdf))]
    ax.bar(rdf["model"],rdf["R2"],color=colors_c,edgecolor="white")
    ax.set_ylim(0,1); ax.set_title(f"R² — {target}")
    ax.tick_params(axis="x",rotation=15)
    for bar,v in zip(ax.patches,rdf["R2"]):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                f"{v:.3f}",ha="center",fontsize=9,fontweight="bold")
plt.tight_layout(); plt.show()


## 8. Conclusiones e Insights

### 💰 Revenue
- **$65.47B en 3 años** con ganancia neta de $17.77B (margen 27.1%)
- El segmento **Premium** genera el mayor revenue pero tarda más en cerrar
- Noviembre y diciembre son los meses de mayor volumen de ventas

### 🏆 Productos
- **Toyota Hilux** y **Ford Ranger** lideran en revenue total
- Los modelos **Luxury** tienen el margen más alto pero el ciclo de venta más largo
- El **segmento Económico** tiene el cierre más rápido (< 5 días promedio)

### 👤 Vendedores
- **Valentina Cruz** y **Sofía Herrera** lideran en ganancia total
- Los vendedores **Senior** tienen mejor margen promedio
- El ratio comisión/ganancia está bien calibrado entre zonas

### 💳 Financiación y leads
- **Contado** genera el ticket más alto pero representa solo 28%
- Los leads de **Referidos** tienen el mayor ticket promedio — priorizar programa de referidos
- **Instagram** genera más volumen que llamadas entrantes

### 🤖 ML
- Predecir el **margen %** es factible con R²~0.31 — el margen depende principalmente del modelo y descuento
- Los **días de cierre** son más difíciles de predecir (alta variabilidad individual)
- Con más variables de comportamiento del cliente se podría mejorar significativamente
